In [1]:
"""
=============================================================================
DATASET CONDENSATION PIPELINE - COMPLETE IMPLEMENTATION (FIXED)
SVM-Based Support Vector Extraction + Smart Sampling
=============================================================================
"""

import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score
from sklearn.datasets import make_classification, load_breast_cancer, load_digits, load_wine, load_iris
from sklearn.neighbors import NearestNeighbors
from scipy.stats import entropy
import warnings
import time

warnings.filterwarnings('ignore')
np.random.seed(42)

# =============================================================================
# STEP 0: DATASET LOADING FUNCTIONS
# =============================================================================

def load_synthetic_dataset(n_samples=10000, n_features=20, n_classes=2, 
                           n_informative=15, n_redundant=5, flip_y=0.05, random_state=42):
    print(f"\n{'='*60}")
    print("LOADING: Synthetic Dataset")
    print(f"{'='*60}")
    X, y = make_classification(
        n_samples=n_samples, n_features=n_features, n_informative=n_informative,
        n_redundant=n_redundant, n_classes=n_classes, n_clusters_per_class=2,
        weights=[0.7, 0.3] if n_classes == 2 else None,
        flip_y=flip_y, random_state=random_state
    )
    print(f"Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    return X, y, f"Synthetic_{n_samples}"

def load_breast_cancer_dataset():
    print(f"\n{'='*60}")
    print("LOADING: Breast Cancer Dataset")
    print(f"{'='*60}")
    data = load_breast_cancer()
    X, y = data.data, data.target
    print(f"Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    return X, y, "Breast_Cancer"

def load_digits_dataset():
    print(f"\n{'='*60}")
    print("LOADING: Digits Dataset")
    print(f"{'='*60}")
    data = load_digits()
    X, y = data.data, data.target
    print(f"Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    return X, y, "Digits"

def load_wine_dataset():
    print(f"\n{'='*60}")
    print("LOADING: Wine Dataset")
    print(f"{'='*60}")
    data = load_wine()
    X, y = data.data, data.target
    print(f"Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    return X, y, "Wine"

def load_iris_dataset():
    print(f"\n{'='*60}")
    print("LOADING: Iris Dataset")
    print(f"{'='*60}")
    data = load_iris()
    X, y = data.data, data.target
    print(f"Samples: {len(X)}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}")
    return X, y, "Iris"

# =============================================================================
# STEP 1: INITIAL SVM TRAINING
# =============================================================================

class SVMTrainer:
    def __init__(self, kernel='rbf', C=1.0, gamma='scale'):
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.model = None
        self.scaler = StandardScaler()
        self._is_fitted = False
        
    def fit(self, X, y):
        print(f"\n{'='*60}")
        print("STEP 1: INITIAL SVM TRAINING")
        print(f"{'='*60}")
        print(f"Kernel: {self.kernel}, C: {self.C}, Gamma: {self.gamma}")
        
        start_time = time.time()
        X_scaled = self.scaler.fit_transform(X)
        self._is_fitted = True
        
        self.model = SVC(
            kernel=self.kernel, C=self.C, gamma=self.gamma,
            probability=True, random_state=42, max_iter=10000
        )
        self.model.fit(X_scaled, y)
        
        train_time = time.time() - start_time
        
        support_indices = self.model.support_
        support_vectors = X[support_indices]
        support_labels = y[support_indices]
        
        # Handle dual coefficients for multi-class
        dual_coefs = self.model.dual_coef_
        if dual_coefs.shape[0] == 1:
            dual_coefs = np.abs(dual_coefs.flatten())
        else:
            dual_coefs = np.abs(dual_coefs)
            # For multi-class, take max across all decision functions
            if dual_coefs.shape[0] > 1:
                dual_coefs = np.max(dual_coefs, axis=0)
            else:
                dual_coefs = dual_coefs.flatten()
        
        print(f"Training time: {train_time:.2f}s")
        print(f"Total SVs extracted: {len(support_indices)}")
        print(f"SV ratio: {len(support_indices)/len(X)*100:.2f}%")
        
        return {
            'model': self.model,
            'scaler': self.scaler,
            'support_indices': support_indices,
            'support_vectors': support_vectors,
            'support_labels': support_labels,
            'dual_coefs': dual_coefs,
            'n_support': len(support_indices),
            'X_scaled': X_scaled,
            'train_time': train_time
        }
    
    def predict(self, X):
        if not self._is_fitted:
            raise ValueError("Trainer not fitted yet!")
        X_scaled = self.scaler.transform(X)
        return self.model.predict(X_scaled)
    
    def predict_proba(self, X):
        if not self._is_fitted:
            raise ValueError("Trainer not fitted yet!")
        X_scaled = self.scaler.transform(X)
        return self.model.predict_proba(X_scaled)
    
    def decision_function(self, X):
        if not self._is_fitted:
            raise ValueError("Trainer not fitted yet!")
        X_scaled = self.scaler.transform(X)
        return self.model.decision_function(X_scaled)

# =============================================================================
# STEP 2: SV VALIDATION & PRUNING
# =============================================================================

class SVPruner:
    def __init__(self):
        self.clean_svs = None
        self.clean_labels = None
        self.pruning_stats = {}
        
    def cross_kernel_consensus(self, X, y, support_indices, 
                               kernels=['rbf', 'poly', 'sigmoid'],
                               consensus_threshold=0.6):
        print(f"\n{'-'*60}")
        print("2.1 Cross-Kernel Consensus SV")
        print(f"{'-'*60}")
        
        if len(X) > 5000:
            subset_idx = np.random.choice(len(X), 5000, replace=False)
            X_sub, y_sub = X[subset_idx], y[subset_idx]
        else:
            X_sub, y_sub = X, y
            subset_idx = np.arange(len(X))
        
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_sub)
        
        sv_votes = np.zeros(len(support_indices))
        kernel_accuracies = {}
        
        for kernel in kernels:
            try:
                svm = SVC(kernel=kernel, C=1.0, gamma='scale', random_state=42, max_iter=5000)
                svm.fit(X_train_s, y_sub)
                
                current_support = svm.support_
                subset_support = subset_idx[current_support]
                
                for i, idx in enumerate(support_indices):
                    if idx in subset_support:
                        sv_votes[i] += 1
                
                acc = svm.score(X_train_s, y_sub)
                kernel_accuracies[kernel] = acc
                print(f"  {kernel:8s}: Accuracy = {acc:.4f}")
            except Exception as e:
                print(f"  {kernel:8s}: Failed ({str(e)})")
                continue
        
        n_valid_kernels = len([k for k in kernels if k in kernel_accuracies])
        if n_valid_kernels > 0:
            sv_votes /= n_valid_kernels
        
        consensus_mask = sv_votes >= consensus_threshold
        consensus_indices = support_indices[consensus_mask]
        
        print(f"  Consensus threshold: {consensus_threshold}")
        print(f"  SVs before: {len(support_indices)}, After: {len(consensus_indices)}")
        
        return consensus_indices, sv_votes
    
    def noise_aware_pruning(self, X, y, support_indices, 
                           k_neighbors=5, noise_threshold=0.4):
        print(f"\n{'-'*60}")
        print("2.2 Noise-Aware SV Pruning")
        print(f"{'-'*60}")
        
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)
        
        sv_data = X_s[support_indices]
        sv_labels = y[support_indices]
        
        n_neighbors = min(k_neighbors + 1, len(X_s))
        nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(X_s)
        distances, indices = nbrs.kneighbors(sv_data)
        
        noise_scores = []
        for i in range(len(support_indices)):
            neighbor_idx = indices[i][1:]
            if len(neighbor_idx) == 0:
                noise_scores.append(0)
                continue
            neighbor_labels = y[neighbor_idx]
            label_consistency = np.mean(neighbor_labels == sv_labels[i])
            noise_scores.append(1 - label_consistency)
        
        noise_scores = np.array(noise_scores)
        clean_mask = noise_scores <= noise_threshold
        clean_indices = support_indices[clean_mask]
        
        print(f"  k_neighbors: {k_neighbors}, Noise threshold: {noise_threshold}")
        print(f"  SVs before: {len(support_indices)}, After: {len(clean_indices)}")
        print(f"  Removed {np.sum(~clean_mask)} noisy SVs")
        
        return clean_indices, noise_scores
    
    def stability_guided_selection(self, X, y, support_indices, 
                                   n_bootstrap=5, stability_threshold=0.5):
        print(f"\n{'-'*60}")
        print("2.3 Stability-Guided SV Selection")
        print(f"{'-'*60}")
        
        scaler = StandardScaler()
        X_s = scaler.fit_transform(X)
        
        n_samples = len(X)
        sv_stability = np.zeros(len(support_indices))
        
        for b in range(n_bootstrap):
            boot_idx = np.random.choice(n_samples, size=n_samples, replace=True)
            X_boot = X_s[boot_idx]
            y_boot = y[boot_idx]
            
            try:
                svm = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=b, max_iter=5000)
                svm.fit(X_boot, y_boot)
                
                boot_support = set(svm.support_)
                
                for i, idx in enumerate(support_indices):
                    matches = np.where(boot_idx == idx)[0]
                    if len(matches) > 0:
                        local_idx = matches[0]
                        if local_idx in boot_support:
                            sv_stability[i] += 1
            except:
                continue
        
        sv_stability /= n_bootstrap
        stable_mask = sv_stability >= stability_threshold
        stable_indices = support_indices[stable_mask]
        
        print(f"  Bootstrap rounds: {n_bootstrap}")
        print(f"  Stability threshold: {stability_threshold}")
        print(f"  SVs before: {len(support_indices)}, After: {len(stable_indices)}")
        print(f"  Mean stability: {np.mean(sv_stability):.3f}")
        
        return stable_indices, sv_stability
    
    def prune_support_vectors(self, X, y, svm_result, 
                            use_consensus=True, use_noise=True, use_stability=True):
        print(f"\n{'='*60}")
        print("STEP 2: SV VALIDATION & PRUNING")
        print(f"{'='*60}")
        
        support_indices = svm_result['support_indices'].copy()
        print(f"Starting with {len(support_indices)} support vectors")
        
        if use_consensus:
            support_indices, _ = self.cross_kernel_consensus(X, y, support_indices)
        if use_noise:
            support_indices, _ = self.noise_aware_pruning(X, y, support_indices)
        if use_stability:
            support_indices, _ = self.stability_guided_selection(X, y, support_indices)
        
        self.clean_svs = X[support_indices]
        self.clean_labels = y[support_indices]
        
        total_removed = svm_result['n_support'] - len(support_indices)
        print(f"\n{'-'*60}")
        print("PRUNING SUMMARY")
        print(f"{'-'*60}")
        print(f"Original SVs:   {svm_result['n_support']}")
        print(f"Clean SVs:      {len(support_indices)}")
        print(f"Removed:        {total_removed} ({total_removed/svm_result['n_support']*100:.1f}%)")
        print(f"Retention:      {len(support_indices)/svm_result['n_support']*100:.1f}%")
        
        return support_indices

# =============================================================================
# STEP 3: SMART SAMPLING FROM REMAINDER
# =============================================================================

class SmartSampler:
    def __init__(self):
        self.sampled_data = None
        self.sampled_labels = None
        self.sampling_stats = {}
        
    def uncertainty_weighted_sampling(self, X_remain, y_remain, svm_trainer, n_samples, temperature=1.0):
        print(f"\n{'-'*60}")
        print("3.1 Uncertainty-Weighted Smart Sampling")
        print(f"{'-'*60}")
        
        decisions = svm_trainer.decision_function(X_remain)
        
        if len(decisions.shape) > 1:
            uncertainties = 1.0 / (np.min(np.abs(decisions), axis=1) + 1e-10)
        else:
            uncertainties = 1.0 / (np.abs(decisions) + 1e-10)
        
        uncertainties = uncertainties ** (1.0 / temperature)
        probs = uncertainties / np.sum(uncertainties)
        
        n_select = min(n_samples, len(X_remain))
        selected_idx = np.random.choice(len(X_remain), size=n_select, replace=False, p=probs)
        
        print(f"  Temperature: {temperature}")
        print(f"  Requested: {n_samples}, Selected: {n_select}")
        print(f"  Mean uncertainty selected: {np.mean(uncertainties[selected_idx]):.4f}")
        print(f"  Mean uncertainty all: {np.mean(uncertainties):.4f}")
        
        return selected_idx
    
    def entropy_boundary_sampling(self, X_remain, y_remain, svm_trainer, n_samples):
        print(f"\n{'-'*60}")
        print("3.2 Information-Theoretic Boundary Entropy Sampling")
        print(f"{'-'*60}")
        
        proba = svm_trainer.predict_proba(X_remain)
        point_entropies = np.array([entropy(p) for p in proba])
        
        decisions = svm_trainer.decision_function(X_remain)
        if len(decisions.shape) > 1:
            min_margins = np.min(np.abs(decisions), axis=1)
        else:
            min_margins = np.abs(decisions)
        
        max_margin = np.max(min_margins) + 1e-10
        norm_margins = min_margins / max_margin
        combined_score = point_entropies * (1.1 - norm_margins)
        
        n_select = min(n_samples, len(X_remain))
        selected_idx = np.argsort(combined_score)[-n_select:]
        
        print(f"  Selected: {n_select}")
        print(f"  Mean entropy selected: {np.mean(point_entropies[selected_idx]):.4f}")
        
        return selected_idx
    
    def sensitivity_guided_sampling(self, X_remain, y_remain, svm_trainer, n_samples, 
                                   n_perturbations=3, perturbation_scale=0.01):
        print(f"\n{'-'*60}")
        print("3.3 Sensitivity-Guided Sampling")
        print(f"{'-'*60}")
        
        X_scaled = svm_trainer.scaler.transform(X_remain)
        base_decisions = svm_trainer.decision_function(X_remain)
        
        if len(base_decisions.shape) > 1:
            base_decisions = np.max(np.abs(base_decisions), axis=1)
        else:
            base_decisions = np.abs(base_decisions)
        
        sensitivities = np.zeros(len(X_remain))
        
        for _ in range(n_perturbations):
            noise = np.random.normal(0, perturbation_scale, X_scaled.shape)
            X_perturbed = X_scaled + noise
            
            perturbed_decisions = svm_trainer.model.decision_function(X_perturbed)
            if len(perturbed_decisions.shape) > 1:
                perturbed_decisions = np.max(np.abs(perturbed_decisions), axis=1)
            else:
                perturbed_decisions = np.abs(perturbed_decisions)
            
            sensitivities += np.abs(base_decisions - perturbed_decisions)
        
        sensitivities /= n_perturbations
        
        n_select = min(n_samples, len(X_remain))
        selected_idx = np.argsort(sensitivities)[-n_select:]
        
        print(f"  Perturbation scale: {perturbation_scale}")
        print(f"  Perturbations: {n_perturbations}")
        print(f"  Selected: {n_select}")
        print(f"  Mean sensitivity selected: {np.mean(sensitivities[selected_idx]):.4f}")
        
        return selected_idx
    
    def sample_remainder(self, X, y, svm_trainer, clean_sv_indices, target_size, method='hybrid'):
        print(f"\n{'='*60}")
        print("STEP 3: SMART SAMPLING FROM REMAINDER")
        print(f"{'='*60}")
        
        all_indices = np.arange(len(X))
        remain_mask = np.ones(len(X), dtype=bool)
        remain_mask[clean_sv_indices] = False
        remain_indices = all_indices[remain_mask]
        
        X_remain = X[remain_indices]
        y_remain = y[remain_mask]
        
        print(f"Remaining data: {len(X_remain)} samples")
        print(f"Target smart samples: {target_size}")
        print(f"Sampling method: {method}")
        
        if method == 'uncertainty':
            selected_local = self.uncertainty_weighted_sampling(X_remain, y_remain, svm_trainer, target_size)
        elif method == 'entropy':
            selected_local = self.entropy_boundary_sampling(X_remain, y_remain, svm_trainer, target_size)
        elif method == 'sensitivity':
            selected_local = self.sensitivity_guided_sampling(X_remain, y_remain, svm_trainer, target_size)
        elif method == 'hybrid':
            n_unc = int(target_size * 0.5)
            n_ent = int(target_size * 0.3)
            n_sens = target_size - n_unc - n_ent
            
            print(f"\n  Hybrid split: Uncertainty={n_unc}, Entropy={n_ent}, Sensitivity={n_sens}")
            
            idx_unc = self.uncertainty_weighted_sampling(X_remain, y_remain, svm_trainer, n_unc)
            idx_ent = self.entropy_boundary_sampling(X_remain, y_remain, svm_trainer, n_ent)
            idx_sens = self.sensitivity_guided_sampling(X_remain, y_remain, svm_trainer, n_sens)
            
            selected_local = np.unique(np.concatenate([idx_unc, idx_ent, idx_sens]))
            
            if len(selected_local) < target_size:
                remaining = list(set(range(len(X_remain))) - set(selected_local))
                if remaining:
                    additional = np.random.choice(remaining, 
                        size=min(target_size - len(selected_local), len(remaining)),
                        replace=False)
                    selected_local = np.concatenate([selected_local, additional])
        else:
            n_select = min(target_size, len(X_remain))
            selected_local = np.random.choice(len(X_remain), n_select, replace=False)
        
        selected_global = remain_indices[selected_local]
        self.sampled_data = X[selected_global]
        self.sampled_labels = y[selected_global]
        
        self.sampling_stats = {
            'method': method,
            'requested': target_size,
            'selected': len(selected_global),
            'from_remainder': len(X_remain)
        }
        
        print(f"\n{'-'*60}")
        print("SAMPLING SUMMARY")
        print(f"{'-'*60}")
        print(f"Smart samples: {len(selected_global)}")
        
        return selected_global

# =============================================================================
# STEP 4: MERGE WITH INTELLIGENT WEIGHTING
# =============================================================================

class DatasetMerger:
    def __init__(self):
        self.reduced_dataset = None
        self.reduced_labels = None
        self.weights = None
        self.merge_stats = {}
    
    def dual_importance_weighting(self, clean_svs, clean_labels, sv_dual_coefs,
                                   smart_samples, smart_labels, svm_trainer):
        print(f"\n{'-'*60}")
        print("4.1 Dual-Importance Distillation")
        print(f"{'-'*60}")
        
        # SV importance from dual coefficients
        if len(sv_dual_coefs) > 0:
            sv_importance = sv_dual_coefs / (np.sum(sv_dual_coefs) + 1e-10)
            sv_importance = sv_importance / (np.max(sv_importance) + 1e-10)
        else:
            sv_importance = np.ones(len(clean_svs))
        
        # Smart sample importance from uncertainty (inverse margin)
        decisions = svm_trainer.decision_function(smart_samples)
        if len(decisions.shape) > 1:
            margins = np.min(np.abs(decisions), axis=1)
        else:
            margins = np.abs(decisions)
        
        smart_importance = 1.0 / (margins + 1e-10)
        smart_importance = smart_importance / (np.max(smart_importance) + 1e-10)
        
        # Combine datasets
        X_combined = np.vstack([clean_svs, smart_samples])
        y_combined = np.concatenate([clean_labels, smart_labels])
        
        # Combine and normalize weights
        weights = np.concatenate([sv_importance, smart_importance])
        weights = weights / np.sum(weights) * len(weights)
        
        print(f"  SV importance: [{sv_importance.min():.3f}, {sv_importance.max():.3f}]")
        print(f"  Smart importance: [{smart_importance.min():.3f}, {smart_importance.max():.3f}]")
        print(f"  Combined weight range: [{weights.min():.3f}, {weights.max():.3f}]")
        
        return X_combined, y_combined, weights
    
    def information_decay_weighting(self, X_combined, y_combined, base_weights, decay_factor=0.98):
        print(f"\n{'-'*60}")
        print("4.2 Information Decay Weighting")
        print(f"{'-'*60}")
        
        sorted_idx = np.argsort(base_weights)[::-1]
        decay_weights = base_weights.copy()
        
        for rank, idx in enumerate(sorted_idx):
            decay_weights[idx] *= (decay_factor ** rank)
        
        decay_weights = decay_weights / np.sum(decay_weights) * len(decay_weights)
        
        print(f"  Decay factor: {decay_factor}")
        print(f"  Weight range: [{decay_weights.min():.3f}, {decay_weights.max():.3f}]")
        
        return decay_weights
    
    def merge_datasets(self, X, y, clean_sv_indices, smart_sample_indices, svm_result, use_decay=True):
        print(f"\n{'='*60}")
        print("STEP 4: MERGE WITH INTELLIGENT WEIGHTING")
        print(f"{'='*60}")
        
        clean_svs = X[clean_sv_indices]
        clean_labels = y[clean_sv_indices]
        
        # Get dual coefficients for clean SVs
        all_dual_coefs = svm_result['dual_coefs']
        sv_idx_map = {idx: i for i, idx in enumerate(svm_result['support_indices'])}
        
        clean_dual_coefs = []
        for idx in clean_sv_indices:
            if idx in sv_idx_map:
                coef_idx = sv_idx_map[idx]
                if coef_idx < len(all_dual_coefs):
                    clean_dual_coefs.append(all_dual_coefs[coef_idx])
                else:
                    clean_dual_coefs.append(1.0)
            else:
                clean_dual_coefs.append(1.0)
        clean_dual_coefs = np.array(clean_dual_coefs)
        
        smart_samples = X[smart_sample_indices]
        smart_labels = y[smart_sample_indices]
        
        # FIX: Use the original fitted trainer from svm_result
        # Create a proper trainer with fitted scaler and model
        trainer = SVMTrainer()
        trainer.scaler = svm_result['scaler']
        trainer.model = svm_result['model']
        trainer._is_fitted = True
        
        # Dual importance weighting using the fitted trainer
        X_combined, y_combined, weights = self.dual_importance_weighting(
            clean_svs, clean_labels, clean_dual_coefs,
            smart_samples, smart_labels, trainer
        )
        
        if use_decay:
            weights = self.information_decay_weighting(X_combined, y_combined, weights)
        
        self.reduced_dataset = X_combined
        self.reduced_labels = y_combined
        self.weights = weights
        
        self.merge_stats = {
            'n_sv': len(clean_svs),
            'n_smart': len(smart_samples),
            'total': len(X_combined),
            'compression_ratio': len(X) / len(X_combined)
        }
        
        print(f"\n{'-'*60}")
        print("MERGE SUMMARY")
        print(f"{'-'*60}")
        print(f"  Clean SVs:        {len(clean_svs)}")
        print(f"  Smart samples:    {len(smart_samples)}")
        print(f"  Total reduced:    {len(X_combined)}")
        print(f"  Original size:    {len(X)}")
        print(f"  Compression:      {len(X)/len(X_combined):.2f}x")
        
        return X_combined, y_combined, weights

# =============================================================================
# STEP 5: VALIDATION & COMPARISON
# =============================================================================

class ModelValidator:
    def __init__(self):
        self.results = {}
        self.comparison_df = None
        
    def train_and_evaluate(self, X_train, y_train, X_test, y_test, dataset_name, sample_weights=None):
        print(f"\n{'='*60}")
        print(f"Training on: {dataset_name}")
        print(f"Training samples: {len(X_train)}")
        print(f"{'='*60}")
        
        models = {
            'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
            'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
            'MLP': MLPClassifier(hidden_layer_sizes=(100, 50), max_iter=1000, random_state=42)
        }
        
        results = {}
        for name, model in models.items():
            start_time = time.time()
            
            if sample_weights is not None:
                model.fit(X_train, y_train, sample_weight=sample_weights)
            else:
                model.fit(X_train, y_train)
            
            train_time = time.time() - start_time
            
            y_pred = model.predict(X_test)
            acc = accuracy_score(y_test, y_pred)
            f1 = f1_score(y_test, y_pred, average='weighted')
            
            results[name] = {
                'accuracy': acc,
                'f1': f1,
                'train_time': train_time,
                'model': model
            }
            
            print(f"  {name:15s} | Acc: {acc:.4f} | F1: {f1:.4f} | Time: {train_time:.2f}s")
        
        self.results[dataset_name] = results
        return results
    
    def compare_datasets(self, X_full, y_full, X_reduced, y_reduced, weights_reduced, test_size=0.2):
        print(f"\n{'='*70}")
        print("STEP 5: VALIDATION - FULL vs REDUCED DATASET")
        print(f"{'='*70}")
        
        X_train_full, X_test, y_train_full, y_test = train_test_split(
            X_full, y_full, test_size=test_size, random_state=42, stratify=y_full
        )
        
        scaler_full = StandardScaler()
        X_train_full_s = scaler_full.fit_transform(X_train_full)
        X_test_s = scaler_full.transform(X_test)
        
        scaler_red = StandardScaler()
        X_train_red_s = scaler_red.fit_transform(X_reduced)
        X_test_red_s = scaler_red.transform(X_test)
        
        print(f"\n{'#'*60}")
        print("FULL DATASET RESULTS")
        print(f"{'#'*60}")
        results_full = self.train_and_evaluate(X_train_full_s, y_train_full, X_test_s, y_test, "FULL_DATASET")
        
        print(f"\n{'#'*60}")
        print("REDUCED DATASET RESULTS")
        print(f"{'#'*60}")
        results_reduced = self.train_and_evaluate(X_train_red_s, y_reduced, X_test_red_s, y_test, 
                                                  "REDUCED_DATASET",
                                                  sample_weights=weights_reduced if weights_reduced is not None else None)
        
        print(f"\n{'='*70}")
        print("COMPARISON SUMMARY")
        print(f"{'='*70}")
        print(f"{'Model':<20} {'Full Acc':<12} {'Reduced Acc':<14} {'Retention':<12} {'Speedup':<10}")
        print("-"*70)
        
        comparison_data = []
        for model_name in results_full.keys():
            full_acc = results_full[model_name]['accuracy']
            red_acc = results_reduced[model_name]['accuracy']
            retention = (red_acc / full_acc) * 100 if full_acc > 0 else 0
            speedup = results_full[model_name]['train_time'] / (results_reduced[model_name]['train_time'] + 1e-10)
            
            print(f"{model_name:<20} {full_acc:<12.4f} {red_acc:<14.4f} {retention:<12.1f}% {speedup:<10.2f}x")
            
            comparison_data.append({
                'Model': model_name,
                'Full_Accuracy': full_acc,
                'Reduced_Accuracy': red_acc,
                'Retention_%': retention,
                'Speedup': speedup
            })
        
        avg_full = np.mean([r['accuracy'] for r in results_full.values()])
        avg_red = np.mean([r['accuracy'] for r in results_reduced.values()])
        avg_f1_full = np.mean([r['f1'] for r in results_full.values()])
        avg_f1_red = np.mean([r['f1'] for r in results_reduced.values()])
        
        print(f"\n{'-'*70}")
        print(f"Average Accuracy Full:    {avg_full:.4f}")
        print(f"Average Accuracy Reduced: {avg_red:.4f}")
        print(f"Overall Retention:        {(avg_red/avg_full)*100:.2f}%")
        print(f"Average F1 Full:          {avg_f1_full:.4f}")
        print(f"Average F1 Reduced:       {avg_f1_red:.4f}")
        
        self.comparison_df = pd.DataFrame(comparison_data)
        
        return results_full, results_reduced

# =============================================================================
# MAIN PIPELINE
# =============================================================================

class DatasetCondensationPipeline:
    def __init__(self, target_reduction=0.1, sampling_method='hybrid'):
        self.target_reduction = target_reduction
        self.sampling_method = sampling_method
        self.pipeline_stats = {}
        
    def run(self, X, y, dataset_name):
        print(f"\n{'#'*70}")
        print(f"DATASET CONDENSATION PIPELINE: {dataset_name}")
        print(f"{'#'*70}")
        print(f"Original size: {len(X)} samples, {X.shape[1]} features")
        print(f"Target reduction: {self.target_reduction*100}%")
        print(f"Target size: ~{int(len(X) * self.target_reduction)} samples")
        
        total_start = time.time()
        
        # Step 1: Train SVM
        svm_trainer = SVMTrainer(kernel='rbf', C=1.0, gamma='scale')
        svm_result = svm_trainer.fit(X, y)
        
        # Step 2: Prune SVs
        pruner = SVPruner()
        clean_sv_indices = pruner.prune_support_vectors(X, y, svm_result)
        
        # Step 3: Smart Sampling
        target_smart = max(int(len(X) * self.target_reduction) - len(clean_sv_indices), 
                          len(clean_sv_indices))
        
        sampler = SmartSampler()
        smart_indices = sampler.sample_remainder(X, y, svm_trainer, clean_sv_indices, 
                                                 target_smart, method=self.sampling_method)
        
        # Step 4: Merge
        merger = DatasetMerger()
        X_reduced, y_reduced, weights = merger.merge_datasets(
            X, y, clean_sv_indices, smart_indices, svm_result
        )
        
        # Step 5: Validate
        validator = ModelValidator()
        results_full, results_reduced = validator.compare_datasets(X, y, X_reduced, y_reduced, weights)
        
        total_time = time.time() - total_start
        
        self.pipeline_stats = {
            'dataset': dataset_name,
            'original_size': len(X),
            'reduced_size': len(X_reduced),
            'compression_ratio': len(X) / len(X_reduced),
            'n_sv': len(clean_sv_indices),
            'n_smart': len(smart_indices),
            'total_time': total_time,
            'avg_accuracy_full': np.mean([r['accuracy'] for r in results_full.values()]),
            'avg_accuracy_reduced': np.mean([r['accuracy'] for r in results_reduced.values()]),
            'retention': np.mean([r['accuracy'] for r in results_reduced.values()]) / 
                        np.mean([r['accuracy'] for r in results_full.values()]) * 100
        }
        
        print(f"\n{'='*70}")
        print("PIPELINE COMPLETE")
        print(f"{'='*70}")
        print(f"Total time: {total_time:.2f}s")
        print(f"Original: {len(X)} → Reduced: {len(X_reduced)} ({len(X)/len(X_reduced):.2f}x compression)")
        print(f"Accuracy retention: {self.pipeline_stats['retention']:.2f}%")
        
        return {
            'X_reduced': X_reduced,
            'y_reduced': y_reduced,
            'weights': weights,
            'stats': self.pipeline_stats,
            'results': (results_full, results_reduced),
            'comparison_df': validator.comparison_df
        }

# =============================================================================
# RUN EXPERIMENTS
# =============================================================================

def run_experiments():
    all_results = []
    
    experiments = [
        (load_synthetic_dataset, {'n_samples': 10000}, 0.1, 'hybrid'),
        (load_synthetic_dataset, {'n_samples': 5000, 'n_features': 50, 'n_classes': 3}, 0.15, 'hybrid'),
        (load_breast_cancer_dataset, {}, 0.2, 'uncertainty'),
        (load_digits_dataset, {}, 0.15, 'hybrid'),
        (load_wine_dataset, {}, 0.25, 'uncertainty'),
        (load_iris_dataset, {}, 0.3, 'hybrid'),
    ]
    
    for loader, kwargs, reduction, method in experiments:
        try:
            if kwargs:
                X, y, name = loader(**kwargs)
            else:
                X, y, name = loader()
            
            pipeline = DatasetCondensationPipeline(target_reduction=reduction, sampling_method=method)
            result = pipeline.run(X, y, name)
            all_results.append(result)
            
        except Exception as e:
            print(f"\nERROR with {name}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    print(f"\n{'#'*70}")
    print("FINAL COMPARISON ACROSS ALL DATASETS")
    print(f"{'#'*70}")
    
    summary_data = []
    for res in all_results:
        stats = res['stats']
        summary_data.append({
            'Dataset': stats['dataset'],
            'Original': stats['original_size'],
            'Reduced': stats['reduced_size'],
            'Compression': f"{stats['compression_ratio']:.2f}x",
            'Full_Acc': f"{stats['avg_accuracy_full']:.4f}",
            'Red_Acc': f"{stats['avg_accuracy_reduced']:.4f}",
            'Retention': f"{stats['retention']:.2f}%"
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.to_string(index=False))
    
    return all_results, summary_df

if __name__ == "__main__":
    print("="*70)
    print("DATASET CONDENSATION PIPELINE")
    print("SVM-Based Support Vector Extraction + Smart Sampling")
    print("="*70)
    
    results, summary = run_experiments()

DATASET CONDENSATION PIPELINE
SVM-Based Support Vector Extraction + Smart Sampling

LOADING: Synthetic Dataset
Samples: 10000, Features: 20, Classes: 2

######################################################################
DATASET CONDENSATION PIPELINE: Synthetic_10000
######################################################################
Original size: 10000 samples, 20 features
Target reduction: 10.0%
Target size: ~1000 samples

STEP 1: INITIAL SVM TRAINING
Kernel: rbf, C: 1.0, Gamma: scale
Training time: 4.25s
Total SVs extracted: 2111
SV ratio: 21.11%

STEP 2: SV VALIDATION & PRUNING
Starting with 2111 support vectors

------------------------------------------------------------
2.1 Cross-Kernel Consensus SV
------------------------------------------------------------
  rbf     : Accuracy = 0.9640
  poly    : Accuracy = 0.9094
  sigmoid : Accuracy = 0.7114
  Consensus threshold: 0.6
  SVs before: 2111, After: 832

------------------------------------------------------------
2.2 No